# Kombinierte Labels: Monatskodierung für die RBF-SVM

**Hypothese:** Die zyklische Darstellung von `Month` beschreibt den saisonalen Kontext besser als die numerische Monatszahl und verbessert dadurch die kombinierte Balanced Accuracy.

**Eingaben:** Skalierte Spektren, One-Hot-kodierte AEZ und eine der zwei offiziell unterstützten Monatsdarstellungen: numerisch oder zyklisch.

**Akzeptanzkriterium:** Ausschließlich die höchste `bacc_combined` auf den fünf Trainingsfolds bestimmt die Monatsdarstellung und die SVM-Hyperparameter. Nur dieser Kandidat wird einmal auf dem gemeinsamen Validierungssplit bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

month_encodings = {
    "numeric": {"label": "numerisch", "use_cyclic_month": False},
    "cyclical": {"label": "zyklisch", "use_cyclic_month": True},
}

## 1. Hyperparameter und Monatsdarstellung suchen

Die offiziell implementierten Darstellungen `numeric` und `cyclical` erhalten dieselbe kleine Suche für `C` und `gamma`. Die Vergleiche nutzen ausschließlich die festen Trainingsfolds; weder der gemeinsame Validierungssplit noch dessen Referenzscore gehen in die Auswahl ein.

In [ ]:
tuned_by_encoding = {}

for encoding_name, encoding_options in month_encodings.items():
    preprocessing = PreprocessingConfig(
        scale=True,
        use_meta=True,
        use_cyclic_month=encoding_options["use_cyclic_month"],
    )
    tuned_by_encoding[encoding_name] = tune(
        make_combined_svm(),
        TuneConfig(
            name=f"combined_svm_month_{encoding_name}_search",
            approach="combined",
            description=(
                "RBF-SVM auf kombinierten Crop/Stage-Labels mit "
                f"{encoding_options['label']}er Month-Kodierung; C und gamma per CV vergleichen."
            ),
            preprocessing=preprocessing,
            param_grid={
                "estimator__C": [30.0, 100.0, 300.0],
                "estimator__gamma": [0.001, 0.003, 0.01],
            },
        ),
    )

cv_summary = pd.DataFrame(
    [
        {
            "Kodierung": encoding_name,
            "Month-Kodierung": encoding_options["label"],
            "CV BAcc kombiniert": tuned_by_encoding[encoding_name].best_score,
            "Beste Parameter": tuned_by_encoding[encoding_name].best_params,
        }
        for encoding_name, encoding_options in month_encodings.items()
    ]
).sort_values("CV BAcc kombiniert", ascending=False)
display(cv_summary)

best_encoding = cv_summary.iloc[0]["Kodierung"]
best_month_encoding = month_encodings[best_encoding]
best_tuned = tuned_by_encoding[best_encoding]

## 2. Einmal validieren

Nur die offizielle Monatsdarstellung mit der höchsten CV-BAcc wird einmal auf dem gemeinsamen Validierungssplit bewertet. Die nicht gewinnende Kodierung erhält keine Holdout-Bewertung.

In [ ]:
best_preprocessing = PreprocessingConfig(
    scale=True,
    use_meta=True,
    use_cyclic_month=best_month_encoding["use_cyclic_month"],
)

result = run(
    best_tuned.best_model,
    RunConfig(
        name="combined_svm_month_encoding",
        approach="combined",
        description=(
            "Beste offizielle Month-Kodierung aus zwei CV-Suchen, einmal auf dem gemeinsamen "
            "Validierungssplit bewertet."
        ),
        preprocessing=best_preprocessing,
        tuning_run=best_tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Auswahl der Monatsdarstellung wird anhand der CV-Tabelle eingeordnet. Der gewählte Kandidat wird gegen die aktuelle RBF-SVM-Referenz verglichen; die Referenz bleibt nur bei einem Validierungsgewinn ersetzt.

In [ ]:
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

REFERENCE_SVM_CV_BACC_COMBINED = 0.8473
REFERENCE_SVM_VALIDATION_BACC_COMBINED = 0.8664

comparison = pd.DataFrame(
    {
        "CV BAcc kombiniert": {
            "RBF-SVM Referenz": REFERENCE_SVM_CV_BACC_COMBINED,
            f"Beste Month-Kodierung ({best_month_encoding['label']})": best_tuned.best_score,
        },
        "Validierung BAcc kombiniert": {
            "RBF-SVM Referenz": REFERENCE_SVM_VALIDATION_BACC_COMBINED,
            f"Beste Month-Kodierung ({best_month_encoding['label']})": result.metrics.bacc_combined,
        },
    }
)
display(comparison)

display(cv_summary)

tuning_ax = plot_tuning_candidates(
    best_tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(
    f"Kombinierte Labels: RBF-SVM mit {best_month_encoding['label']}er Month-Kodierung"
)
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
{
    "tuning_runs": {
        encoding_name: tuned.run_id
        for encoding_name, tuned in tuned_by_encoding.items()
    },
    "best_month_encoding": best_encoding,
    "best_month_encoding_label": best_month_encoding["label"],
    "best_cv_score": best_tuned.best_score,
    "best_params": best_tuned.best_params,
    "evaluation_run": result.run_id,
    "metrics": result.metrics.model_dump(),
}

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

tuning_ax = plot_tuning_candidates(
    best_tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(
    f"Kombinierte Labels: RBF-SVM mit {best_month_encoding['label']}er Month-Kodierung"
)
save_doc_figure(tuning_ax.figure, "combined_svm_month_encoding_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_svm_month_encoding_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_svm_month_encoding_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_svm_month_encoding_confusions", MODEL_DOCS_FIGURES_DIR)

## Fazit

Die numerische Darstellung von `Month` gewinnt die CV gegen die zyklische Kodierung (BAcc kombiniert 0.8517 gegenüber 0.8494). Ihr einmaliges Validierungsergebnis von 0.8585 liegt jedoch unter der bestehenden RBF-SVM-Referenz von 0.8664. Die Hypothese wird daher nicht gestützt; die RBF-SVM-Referenz bleibt bestehen.